<h1 style="color: #1E90FF; font-size: 2.5em; font-weight: bold;">
    FIFA 21 Player Market Value — ML Regression Models
</h1>

### <font color='#1E90FF'>**Table of Contents**</font> <a class="anchor" id='toc'></a>

- [1. Setup & Load Data](#1)
- [2. Feature Selection & Preprocessing](#2)
- [3. Train/Test Split](#3)
- [4. Models](#4)
    - [4.1. Linear Regression](#4_1)
    - [4.2. Random Forest](#4_2)
    - [4.3. Gradient Boosted Trees (GBT)](#4_3)
- [5. Hyperparameter Tuning with CrossValidator](#5)
    - [5.1. Tuning Random Forest](#5_1)
    - [5.2. Tuning GBT](#5_2)
- [6. Model Comparison](#6)
    - [6.1. Model Comparison](#6_1)
- [7. Feature Importance](#7)

<a class="anchor" id="1"></a>

# **1. Setup & Load Data**

[Back to TOC](#toc)

In [24]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, log1p, when, expm1
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
)
from pyspark.ml.regression import (
    LinearRegression, RandomForestRegressor, GBTRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = SparkSession.builder \
    .master("local[4]") \
    .appName("FIFA21 ML Models") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark Session ready!")

Spark Session ready!


In [2]:
fifa = spark.read.parquet("cleaned_fifa.parquet")
fifa.createOrReplaceTempView("fifa")

print(f"Rows: {fifa.count()}")
print(f"Columns: {len(fifa.columns)}")
fifa.printSchema()

26/06/01 15:51:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Rows: 18979
Columns: 75
root
 |-- ID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Nationality: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- OVA: integer (nullable = true)
 |-- POT: integer (nullable = true)
 |-- Club: string (nullable = true)
 |-- Positions: string (nullable = true)
 |-- Height: integer (nullable = true)
 |-- Weight: integer (nullable = true)
 |-- Preferred_Foot: string (nullable = true)
 |-- BOV: integer (nullable = true)
 |-- Best_Position: string (nullable = true)
 |-- Value: integer (nullable = true)
 |-- Wage: integer (nullable = true)
 |-- Release_Clause: integer (nullable = true)
 |-- Attacking: integer (nullable = true)
 |-- Crossing: integer (nullable = true)
 |-- Finishing: integer (nullable = true)
 |-- Heading_Accuracy: integer (nullable = true)
 |-- Short_Passing: integer (nullable = true)
 |-- Volleys: integer (nullable = true)
 |-- Skill: integer (nullable = true)
 |-- Dribbling: integer (nullable = true)
 |-- 

<a class="anchor" id="2"></a>

# **2. Feature Selection & Preprocessing**

[Back to TOC](#toc)

We select features that are available at prediction time (i.e., no data leakage from financials like `Wage` or `Release Clause`).  
The target variable `Value` is **log-transformed** (`log1p`) to reduce the effect of the extreme right skew observed during EDA — this is standard practice for monetary targets.

In [3]:
# --- Log-transform the target to handle right skew ---
# Value = 0 players (free agents / no club) are excluded — they have no market value to predict
fifa_ml = fifa.filter(col("Value") > 0).withColumn("log_Value", log1p(col("Value")))

print(f"Rows after filtering Value > 0: {fifa_ml.count()}")

Rows after filtering Value > 0: 18731


In [4]:
# --- Define feature groups ---

# Numeric features (no leakage: excluding Wage, Release_Clause, Value itself)
numeric_features = [
    "Age", "OVA", "POT", "BOV",
    "Height", "Weight",
    "Attacking", "Crossing", "Finishing", "Heading_Accuracy",
    "Short_Passing", "Volleys",
    "Skill", "Dribbling", "Curve", "FK_Accuracy",
    "Long_Passing", "Ball_Control",
    "Movement", "Acceleration", "Sprint_Speed", "Agility",
    "Reactions", "Balance",
    "Power", "Shot_Power", "Jumping", "Stamina", "Strength", "Long_Shots",
    "Mentality", "Aggression", "Interceptions", "Positioning",
    "Vision", "Penalties", "Composure",
    "Defending", "Marking", "Standing_Tackle", "Sliding_Tackle",
    "Goalkeeping", "GK_Diving", "GK_Handling", "GK_Kicking",
    "GK_Positioning", "GK_Reflexes",
    "Total_Stats", "Base_Stats",
    "WF", "SM", "IR",
    "N_Positions"
]

# Categorical features to encode
categorical_features = ["Preferred_Foot", "Contract_Status"]

# Keep only columns that exist in the DataFrame
existing_cols = set(fifa_ml.columns)
numeric_features  = [f for f in numeric_features  if f in existing_cols]
categorical_features = [f for f in categorical_features if f in existing_cols]

print(f"Numeric features selected : {len(numeric_features)}")
print(f"Categorical features selected: {len(categorical_features)}")

Numeric features selected : 50
Categorical features selected: 2


In [5]:
# --- Drop rows with nulls in selected features or target ---
all_features = numeric_features + categorical_features
fifa_ml = fifa_ml.dropna(subset=all_features + ["log_Value"])

print(f"Rows after dropping nulls: {fifa_ml.count()}")

Rows after dropping nulls: 18731


In [6]:
# --- Build encoding stages for categorical features ---
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]

ohe_cols = [f"{c}_ohe" for c in categorical_features]
all_assembled_cols = numeric_features + ohe_cols

# --- VectorAssembler + StandardScaler ---
assembler = VectorAssembler(
    inputCols=all_assembled_cols,
    outputCol="raw_features",
    handleInvalid="skip"
)

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

print("Preprocessing pipeline stages defined.")

Preprocessing pipeline stages defined.


<a class="anchor" id="3"></a>

# **3. Train / Test Split**

[Back to TOC](#toc)

We use an 80/20 stratified-equivalent random split with a fixed seed for reproducibility.

In [7]:
train_df, test_df = fifa_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Train rows : {train_df.count()}")
print(f"Test rows  : {test_df.count()}")

Train rows : 15050
Test rows  : 3681


<a class="anchor" id="4"></a>

# **4. Models (Baseline)**

[Back to TOC](#toc)

We train three baseline models with default hyperparameters first. All models predict `log_Value`; predictions are exponentiated (`expm1`) to report errors in the original €-scale.

In [25]:
# --- Helper: evaluate a fitted pipeline on test set ---
from pyspark.sql.functions import expm1 as spark_expm1

evaluator_rmse = RegressionEvaluator(labelCol="log_Value", predictionCol="prediction", metricName="rmse")
evaluator_r2   = RegressionEvaluator(labelCol="log_Value", predictionCol="prediction", metricName="r2")
evaluator_mae  = RegressionEvaluator(labelCol="log_Value", predictionCol="prediction", metricName="mae")

results = {}  # store model name -> metrics dict

def evaluate_model(name, pipeline_model, test_data):
    preds = pipeline_model.transform(test_data)
    rmse_log = evaluator_rmse.evaluate(preds)
    r2       = evaluator_r2.evaluate(preds)
    mae_log  = evaluator_mae.evaluate(preds)
    preds_euro = preds.withColumn("Value_Euro", expm1(col("log_Value"))) \
                      .withColumn("Prediction_Euro", expm1(col("prediction")))
    evaluator_mae_euro = RegressionEvaluator(labelCol="Value_Euro", predictionCol="Prediction_Euro", metricName="mae")
    mae_euro = evaluator_mae_euro.evaluate(preds_euro)
    results[name] = {
        "R²": round(r2, 4), 
        "RMSE (log)": round(rmse_log, 4), 
        "MAE (log)": round(mae_log, 4),
        "MAE (€)": round(mae_euro, 2)  
    }    
    print(f"\n{'='*40}")
    print(f"Model : {name}")
    print(f"  R²              : {r2:.4f}")
    print(f"  RMSE (log-scale): {rmse_log:.4f}")
    print(f"  MAE  (log-scale): {mae_log:.4f}")
    print(f"  MAE  (Euros €)  : {mae_euro:,.2f} €") 
    return preds

<a class="anchor" id="4_1"></a>

## **4.1. Linear Regression**

[Back to TOC](#toc)

In [26]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="log_Value",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0  # Ridge (L2)
)

pipeline_lr = Pipeline(stages=indexers + encoders + [assembler, scaler, lr])
model_lr    = pipeline_lr.fit(train_df)

preds_lr = evaluate_model("Linear Regression (baseline)", model_lr, test_df)


Model : Linear Regression (baseline)
  R²              : 0.9633
  RMSE (log-scale): 0.2360
  MAE  (log-scale): 0.1688
  MAE  (Euros €)  : 662,762.31 €


In [10]:
# Training summary
lr_summary = model_lr.stages[-1].summary
print(f"Training RMSE : {lr_summary.rootMeanSquaredError:.4f}")
print(f"Training R²   : {lr_summary.r2:.4f}")

Training RMSE : 0.2394
Training R²   : 0.9630


<a class="anchor" id="4_2"></a>

## **4.2. Random Forest**

[Back to TOC](#toc)

In [27]:
rf = RandomForestRegressor(
    featuresCol="raw_features",  # tree-based models don't need scaling
    labelCol="log_Value",
    numTrees=100,
    maxDepth=8,
    seed=42
)

# Note: StandardScaler is NOT added for tree-based models — it has no effect
pipeline_rf = Pipeline(stages=indexers + encoders + [assembler, rf])
model_rf    = pipeline_rf.fit(train_df)

preds_rf = evaluate_model("Random Forest (baseline)", model_rf, test_df)

26/06/01 17:18:43 WARN DAGScheduler: Broadcasting large task binary with size 1389.9 KiB
26/06/01 17:18:45 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/06/01 17:18:47 WARN DAGScheduler: Broadcasting large task binary with size 4.7 MiB
26/06/01 17:18:50 WARN DAGScheduler: Broadcasting large task binary with size 1187.5 KiB



Model : Random Forest (baseline)
  R²              : 0.9884
  RMSE (log-scale): 0.1327
  MAE  (log-scale): 0.0957
  MAE  (Euros €)  : 335,034.65 €


<a class="anchor" id="4_3"></a>

## **4.3. Gradient Boosted Trees (GBT)**

[Back to TOC](#toc)

In [28]:
gbt = GBTRegressor(
    featuresCol="raw_features",
    labelCol="log_Value",
    maxIter=100,
    maxDepth=5,
    stepSize=0.1,
    seed=42
)

pipeline_gbt = Pipeline(stages=indexers + encoders + [assembler, gbt])
model_gbt    = pipeline_gbt.fit(train_df)

preds_gbt = evaluate_model("GBT (baseline)", model_gbt, test_df)


Model : GBT (baseline)
  R²              : 0.9952
  RMSE (log-scale): 0.0853
  MAE  (log-scale): 0.0580
  MAE  (Euros €)  : 215,425.02 €


<a class="anchor" id="5"></a>

# **5. Hyperparameter Tuning with CrossValidator**

[Back to TOC](#toc)

We use 5-fold `CrossValidator` for the two best-performing tree models. Linear Regression is simpler and its regularisation is tuned inline. Grid search explores a controlled space to balance compute time and coverage.

<a class="anchor" id="5_1"></a>

## **5.1. Tuning Random Forest**

[Back to TOC](#toc)

In [13]:
rf_tuned = RandomForestRegressor(
    featuresCol="raw_features",
    labelCol="log_Value",
    seed=42
)

pipeline_rf_tune = Pipeline(stages=indexers + encoders + [assembler, rf_tuned])

param_grid_rf = (
    ParamGridBuilder()
    .addGrid(rf_tuned.numTrees,  [50, 100, 200])
    .addGrid(rf_tuned.maxDepth,  [6, 8, 10])
    .addGrid(rf_tuned.minInstancesPerNode, [1, 5])
    .build()
)

cv_rf = CrossValidator(
    estimator=pipeline_rf_tune,
    estimatorParamMaps=param_grid_rf,
    evaluator=evaluator_rmse,
    numFolds=5,
    seed=42
)

print(f"Fitting RF CrossValidator — {len(param_grid_rf)} param combos × 5 folds ...")
cv_model_rf = cv_rf.fit(train_df)

best_rf_params = cv_model_rf.bestModel.stages[-1].extractParamMap()
print("\nBest RF parameters:")
for param, val in best_rf_params.items():
    print(f"  {param.name}: {val}")

Fitting RF CrossValidator — 18 param combos × 5 folds ...


26/06/01 15:52:38 WARN DAGScheduler: Broadcasting large task binary with size 1430.3 KiB
26/06/01 15:52:39 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/06/01 15:52:43 WARN DAGScheduler: Broadcasting large task binary with size 1425.7 KiB
26/06/01 15:52:44 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/06/01 15:52:48 WARN DAGScheduler: Broadcasting large task binary with size 1430.3 KiB
26/06/01 15:52:49 WARN DAGScheduler: Broadcasting large task binary with size 2.5 MiB
26/06/01 15:52:50 WARN DAGScheduler: Broadcasting large task binary with size 4.5 MiB
26/06/01 15:52:51 WARN DAGScheduler: Broadcasting large task binary with size 1054.7 KiB
26/06/01 15:52:52 WARN DAGScheduler: Broadcasting large task binary with size 7.8 MiB
26/06/01 15:52:54 WARN DAGScheduler: Broadcasting large task binary with size 1721.3 KiB
26/06/01 15:52:58 WARN DAGScheduler: Broadcasting large task binary with size 1425.7 KiB
26/06/01 15:52:59 WARN DAGScheduler:


Best RF parameters:
  bootstrap: True
  cacheNodeIds: False
  checkpointInterval: 10
  featureSubsetStrategy: auto
  featuresCol: raw_features
  impurity: variance
  labelCol: log_Value
  leafCol: 
  maxBins: 32
  maxDepth: 10
  maxMemoryInMB: 256
  minInfoGain: 0.0
  minInstancesPerNode: 1
  minWeightFractionPerNode: 0.0
  numTrees: 200
  predictionCol: prediction
  seed: 42
  subsamplingRate: 1.0


In [29]:
preds_rf_tuned = evaluate_model("Random Forest (tuned)", cv_model_rf, test_df)


Model : Random Forest (tuned)
  R²              : 0.9926
  RMSE (log-scale): 0.1062
  MAE  (log-scale): 0.0753
  MAE  (Euros €)  : 273,188.59 €


<a class="anchor" id="5_2"></a>

## **5.2. Tuning GBT**

[Back to TOC](#toc)

In [15]:
gbt_tuned = GBTRegressor(
    featuresCol="raw_features",
    labelCol="log_Value",
    seed=42
)

pipeline_gbt_tune = Pipeline(stages=indexers + encoders + [assembler, gbt_tuned])

param_grid_gbt = (
    ParamGridBuilder()
    .addGrid(gbt_tuned.maxIter,  [50, 100, 150])
    .addGrid(gbt_tuned.maxDepth, [4, 5, 6])
    .addGrid(gbt_tuned.stepSize, [0.05, 0.1, 0.2])
    .build()
)

cv_gbt = CrossValidator(
    estimator=pipeline_gbt_tune,
    estimatorParamMaps=param_grid_gbt,
    evaluator=evaluator_rmse,
    numFolds=5,
    seed=42
)

print(f"Fitting GBT CrossValidator — {len(param_grid_gbt)} param combos × 5 folds ...")
cv_model_gbt = cv_gbt.fit(train_df)

best_gbt_params = cv_model_gbt.bestModel.stages[-1].extractParamMap()
print("\nBest GBT parameters:")
for param, val in best_gbt_params.items():
    print(f"  {param.name}: {val}")

Fitting GBT CrossValidator — 27 param combos × 5 folds ...


26/06/01 16:18:12 WARN DAGScheduler: Broadcasting large task binary with size 1001.3 KiB
26/06/01 16:18:12 WARN DAGScheduler: Broadcasting large task binary with size 1001.8 KiB
26/06/01 16:18:12 WARN DAGScheduler: Broadcasting large task binary with size 1002.4 KiB
26/06/01 16:18:12 WARN DAGScheduler: Broadcasting large task binary with size 1003.6 KiB
26/06/01 16:18:12 WARN DAGScheduler: Broadcasting large task binary with size 1005.9 KiB
26/06/01 16:19:23 WARN DAGScheduler: Broadcasting large task binary with size 1001.7 KiB
26/06/01 16:19:23 WARN DAGScheduler: Broadcasting large task binary with size 1004.3 KiB
26/06/01 16:19:23 WARN DAGScheduler: Broadcasting large task binary with size 1004.8 KiB
26/06/01 16:19:23 WARN DAGScheduler: Broadcasting large task binary with size 1005.3 KiB
26/06/01 16:19:23 WARN DAGScheduler: Broadcasting large task binary with size 1006.5 KiB
26/06/01 16:19:23 WARN DAGScheduler: Broadcasting large task binary with size 1008.8 KiB
26/06/01 16:19:23 WAR


Best GBT parameters:
  cacheNodeIds: False
  checkpointInterval: 10
  featureSubsetStrategy: all
  featuresCol: raw_features
  impurity: variance
  labelCol: log_Value
  leafCol: 
  lossType: squared
  maxBins: 32
  maxDepth: 4
  maxIter: 150
  maxMemoryInMB: 256
  minInfoGain: 0.0
  minInstancesPerNode: 1
  minWeightFractionPerNode: 0.0
  predictionCol: prediction
  seed: 42
  stepSize: 0.2
  subsamplingRate: 1.0
  validationTol: 0.01


In [30]:
preds_gbt_tuned = evaluate_model("GBT (tuned)", cv_model_gbt, test_df)


Model : GBT (tuned)
  R²              : 0.9963
  RMSE (log-scale): 0.0749
  MAE  (log-scale): 0.0535
  MAE  (Euros €)  : 203,206.95 €


<a class="anchor" id="6"></a>

# **6. Model Comparison**

[Back to TOC](#toc)

In [17]:
results_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"})
print(results_df.to_string(index=False))

                       Model  RMSE (log)     R²  MAE (log)
Linear Regression (baseline)      0.2360 0.9633     0.1688
    Random Forest (baseline)      0.1327 0.9884     0.0957
              GBT (baseline)      0.0853 0.9952     0.0580
       Random Forest (tuned)      0.1062 0.9926     0.0753
                 GBT (tuned)      0.0749 0.9963     0.0535


In [18]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["RMSE (log-scale) ↓", "MAE (log-scale) ↓", "R² ↑"]
)

colors = px.colors.qualitative.Set2[:len(results_df)]

for metric, col_idx in [("RMSE (log)", 1), ("MAE (log)", 2), ("R²", 3)]:
    fig.add_trace(
        go.Bar(
            x=results_df["Model"],
            y=results_df[metric],
            name=metric,
            marker_color=colors,
            text=results_df[metric].round(4),
            textposition="outside"
        ),
        row=1, col=col_idx
    )

fig.update_layout(
    title="Model Comparison — Test Set Metrics",
    showlegend=False,
    height=450
)
fig.update_xaxes(tickangle=-30)
fig.show()

In [19]:
# Predicted vs Actual scatter for best model (GBT tuned)
best_preds = preds_gbt_tuned.select("log_Value", "prediction").toPandas()  # NOTE: sampled for viz

fig = px.scatter(
    best_preds.sample(min(3000, len(best_preds)), random_state=42),
    x="log_Value", y="prediction",
    opacity=0.4,
    labels={"log_Value": "Actual log(Value)", "prediction": "Predicted log(Value)"},
    title="GBT (tuned) — Predicted vs Actual (log scale)"
)

# Perfect prediction line
min_v = best_preds["log_Value"].min()
max_v = best_preds["log_Value"].max()
fig.add_trace(go.Scatter(
    x=[min_v, max_v], y=[min_v, max_v],
    mode="lines", name="Perfect",
    line=dict(color="red", dash="dash")
))

fig.show()

In [20]:
# Residual plot
best_preds["residual"] = best_preds["prediction"] - best_preds["log_Value"]

fig = px.scatter(
    best_preds.sample(min(3000, len(best_preds)), random_state=42),
    x="prediction", y="residual",
    opacity=0.4,
    labels={"prediction": "Predicted log(Value)", "residual": "Residual"},
    title="GBT (tuned) — Residuals vs Predicted"
)
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

<a class="anchor" id="6_1"></a>

# **6.1. Overfitting Analysis (Train vs Test)**

[Back to TOC](#toc)


In [1]:
# 1. Recolher todos os modelos finais num dicionário
# (Usamos os base e os otimizados pelo CrossValidator)
trained_models = {
    "Linear Reg": model_lr,
    "RF (Base)": model_rf,
    "GBT (Base)": model_gbt,
    "RF (Tuned)": cv_model_rf.bestModel,
    "GBT (Tuned)": cv_model_gbt.bestModel
}

overfit_data = []

# 2. Calcular RMSE para Treino e Teste em cada modelo
for name, model in trained_models.items():
    # Prever no Treino
    preds_train = model.transform(train_df)
    rmse_train = evaluator_rmse.evaluate(preds_train)
    
    # Prever no Teste
    preds_test = model.transform(test_df)
    rmse_test = evaluator_rmse.evaluate(preds_test)
    
    # Guardar os resultados no formato longo (ideal para o Plotly)
    overfit_data.append({"Model": name, "Dataset": "Train", "RMSE": rmse_train})
    overfit_data.append({"Model": name, "Dataset": "Test", "RMSE": rmse_test})

# Converter para Pandas para visualização
overfit_df = pd.DataFrame(overfit_data)

# 3. Criar o gráfico de barras agrupadas
fig = px.bar(
    overfit_df, 
    x="Model", 
    y="RMSE", 
    color="Dataset", 
    barmode="group",
    title="Overfitting Analysis: Train vs Test RMSE (log-scale)",
    labels={"RMSE": "RMSE (Lower is better)", "Model": "Model"},
    color_discrete_map={"Train": "royalblue", "Test": "darkorange"},
    text="RMSE"
)

# Melhorar o aspeto do gráfico
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside', marker_line_color='black', marker_line_width=1)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=500, template="plotly_white")
fig.show()

NameError: name 'model_lr' is not defined

<a class="anchor" id="7"></a>

# **7. Feature Importance**

[Back to TOC](#toc)

Random Forest and GBT both expose `featureImportances`. We use the tuned RF model here as it provides stable importance estimates.

In [21]:
best_rf_model = cv_model_rf.bestModel.stages[-1]  # RandomForestRegressionModel

# Reconstruct feature names after OHE
# OHE adds N-1 columns per categorical; for simplicity we label them with _ohe_0, _ohe_1, ...
ohe_feature_names = []
for c in categorical_features:
    ohe_model = cv_model_rf.bestModel.stages[
        len(indexers) + categorical_features.index(c)
    ]
    n_cats = ohe_model.categorySizes[0] - 1  # drop last for OHE
    ohe_feature_names += [f"{c}_{i}" for i in range(n_cats)]

feature_names = numeric_features + ohe_feature_names

importances = best_rf_model.featureImportances.toArray()

# Pad / trim to match length if needed
min_len = min(len(feature_names), len(importances))
feat_imp_df = pd.DataFrame({
    "Feature": feature_names[:min_len],
    "Importance": importances[:min_len]
}).sort_values("Importance", ascending=False).head(25)

fig = px.bar(
    feat_imp_df,
    x="Importance", y="Feature",
    orientation="h",
    title="Top 25 Feature Importances — Random Forest (tuned)",
    color="Importance", color_continuous_scale="Blues",
    text="Importance"
)
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.show()

In [22]:
# GBT feature importances
best_gbt_model = cv_model_gbt.bestModel.stages[-1]

gbt_importances = best_gbt_model.featureImportances.toArray()
min_len_gbt = min(len(feature_names), len(gbt_importances))

gbt_imp_df = pd.DataFrame({
    "Feature": feature_names[:min_len_gbt],
    "Importance": gbt_importances[:min_len_gbt]
}).sort_values("Importance", ascending=False).head(25)

fig = px.bar(
    gbt_imp_df,
    x="Importance", y="Feature",
    orientation="h",
    title="Top 25 Feature Importances — GBT (tuned)",
    color="Importance", color_continuous_scale="Greens",
    text="Importance"
)
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.show()

In [23]:
# Final summary table
print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)
print(results_df.sort_values("R²", ascending=False).to_string(index=False))
print("="*60)
print("Note: all metrics computed on log1p(Value). Lower RMSE/MAE and higher R² is better.")


FINAL MODEL COMPARISON
                       Model  RMSE (log)     R²  MAE (log)
                 GBT (tuned)      0.0749 0.9963     0.0535
              GBT (baseline)      0.0853 0.9952     0.0580
       Random Forest (tuned)      0.1062 0.9926     0.0753
    Random Forest (baseline)      0.1327 0.9884     0.0957
Linear Regression (baseline)      0.2360 0.9633     0.1688
Note: all metrics computed on log1p(Value). Lower RMSE/MAE and higher R² is better.
